# 🎬 Movie Recommendation System

A **content-based movie recommendation system** built using the  `movies_20000.xlsx` dataset.


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load the attached dataset
file_path = "movies_20000.xlsx"
movies = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Shape:", movies.shape)
print("Columns:", movies.columns.tolist())

movies.head()

In [ ]:
# Basic dataset inspection
print("Number of movies:", len(movies))
print("\nMissing values:")
print(movies.isnull().sum())

print("\nDuplicate rows:", movies.duplicated().sum())

print("\nData types:")
print(movies.dtypes)

In [ ]:
# Clean the data
movies = movies[["title", "description"]].copy()

movies["title"] = movies["title"].fillna("").astype(str).str.strip()
movies["description"] = movies["description"].fillna("").astype(str).str.strip()

movies = movies[
    (movies["title"] != "") &
    (movies["description"] != "")
].reset_index(drop=True)

print("Dataset after cleaning:", movies.shape)
movies.head()

## 🔎 Exploratory Data Analysis

The dataset contains movie titles and textual descriptions. Since this is a **content-based recommender**, the description is the main feature used to determine movie similarity.

In [ ]:
# Description length analysis
movies["description_length"] = movies["description"].str.len()

print(movies["description_length"].describe())

plt.figure(figsize=(10, 5))
plt.hist(movies["description_length"], bins=40)
plt.title("Distribution of Movie Description Length")
plt.xlabel("Description Length")
plt.ylabel("Number of Movies")
plt.show()

movies = movies.drop(columns=["description_length"])

## 🧠 TF-IDF Vectorization

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts each movie description into a numerical vector.

Words that are common across many movies receive less importance, while words that are more specific to a movie receive higher importance.

In [ ]:
# Create TF-IDF matrix from movie descriptions
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(movies["description"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

## 📐 Cosine Similarity

Cosine similarity measures how similar two movie-description vectors are.

For a selected movie, we compare its TF-IDF vector against all movie vectors and rank the results from highest similarity to lowest similarity.

In [ ]:
# Create a fast title lookup
title_to_indices = {}

for idx, title in enumerate(movies["title"]):
    key = title.lower().strip()
    title_to_indices.setdefault(key, []).append(idx)

print("Unique title keys:", len(title_to_indices))

In [ ]:
def recommend_movies(movie_title, number_of_recommendations=5):
    """Return movies whose descriptions are most similar to the selected movie."""

    movie_title = str(movie_title).strip()
    key = movie_title.lower()

    if key not in title_to_indices:
        return f"Movie '{movie_title}' was not found in the dataset."

    # If duplicate titles exist, use the first occurrence.
    movie_index = title_to_indices[key][0]

    # Compare only the selected movie with the complete TF-IDF matrix.
    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    ranked_indices = np.argsort(similarity_scores)[::-1]

    recommendations = []

    for idx in ranked_indices:
        if idx == movie_index:
            continue

        recommendations.append({
            "title": movies.iloc[idx]["title"],
            "similarity_score": round(float(similarity_scores[idx]), 4)
        })

        if len(recommendations) == number_of_recommendations:
            break

    return pd.DataFrame(recommendations)

## 🎯 Test the Recommendation System

Try any movie title from the dataset. The examples below use titles from the attached dataset.

In [ ]:
recommend_movies("Interstellar", 5)

In [ ]:
recommend_movies("Iron Man", 5)

In [ ]:
recommend_movies("Before Cairo", 5)

In [ ]:
recommend_movies("Code of A Floating City", 5)

In [ ]:
# Example of a title that does not exist
recommend_movies("This Movie Does Not Exist", 5)

## 🔍 Search for a Movie Title

Use this helper to find titles containing a keyword before requesting recommendations.

In [ ]:
def search_movies(keyword, limit=20):
    keyword = str(keyword).strip().lower()

    results = movies[
        movies["title"].str.lower().str.contains(keyword, na=False)
    ][["title"]].head(limit)

    return results.reset_index(drop=True)

search_movies("dark")

## 🎬 Interactive Recommendation

Enter a movie title and choose how many recommendations you want.

In [ ]:
movie_name = input("Enter a movie title: ")
n = input("Number of recommendations (default 5): ").strip()

n = int(n) if n.isdigit() and int(n) > 0 else 5

result = recommend_movies(movie_name, n)

print("\nRecommended Movies:")
display(result)

## 📊 How the System Works

**Movie Description → TF-IDF → Cosine Similarity → Ranked Recommendations**

1. **Input:** A movie title.
2. **Feature extraction:** The movie description is transformed using TF-IDF.
3. **Similarity calculation:** Cosine similarity compares the selected movie with all other movies.
4. **Ranking:** Movies are sorted by similarity score.
5. **Output:** The top similar movies are recommended.

### Why content-based filtering?
The dataset contains movie descriptions but does not contain user ratings or user-item interactions. Therefore, content-based filtering is appropriate for this dataset.

### Important improvement over the reference notebook
The reference notebook computes a complete similarity matrix. For 20,000 movies, that creates a very large dense matrix. This notebook instead computes:

`cosine_similarity(selected_movie_vector, tfidf_matrix)`

only when a recommendation is requested, which substantially reduces memory usage.